In [ ]:
import numpy as np
import camb
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt



def generate_mock_catalogue(
    z,
    box_size=1000.0,      # Mpc/h
    ngrid=128,
    nbar=1e-6,            # galaxies/(Mpc/h)^3
    bias=1.5,
    seed=None,
):
    """
    Generate a simple Gaussian mock galaxy catalogue.

    Parameters
    ----------
    z : float
        Redshift.
    box_size : float
        Side length of cubic box [Mpc/h].
    ngrid : int
        Number of grid cells per dimension.
    nbar : float
        Mean galaxy density [(Mpc/h)^-3].
    bias : float
        Linear galaxy bias.
    seed : int or None
        Random seed.

    Returns
    -------
    positions : ndarray
        Galaxy positions, shape (Ngal, 3), in Mpc/h.
    delta : ndarray
        Density contrast field.
    """

    rng = np.random.default_rng(seed)

    # ------------------------------------------------------------------
    # CAMB matter power spectrum
    # ------------------------------------------------------------------

    pars = camb.CAMBparams()

    pars.NonLinear = model.NonLinear_both

    pars.set_cosmology(
        H0=67.4,
        ombh2=0.0224,
        omch2=0.120
    )

    pars.InitPower.set_params(
        As=2.1e-9,
        ns=0.965
    )

    kmax = 20.0

    pars.set_matter_power(
        redshifts=[z],
        kmax=kmax
    )

    results = camb.get_results(pars)

    kh, zs, pk = results.get_matter_power_spectrum(
        minkh=1e-4,
        maxkh=kmax,
        npoints=2000
    )

    pk = pk[0]

    # ------------------------------------------------------------------
    # Fourier grid
    # ------------------------------------------------------------------

    dx = box_size / ngrid

    kfreq = np.fft.fftfreq(ngrid, d=dx) * 2 * np.pi

    kx, ky, kz = np.meshgrid(
        kfreq,
        kfreq,
        kfreq,
        indexing="ij"
    )

    k = np.sqrt(kx**2 + ky**2 + kz**2)

    pk_interp = interp1d(
        kh,
        pk,
        bounds_error=False,
        fill_value=0.0
    )

    Pk_grid = pk_interp(k)

    # ------------------------------------------------------------------
    # Gaussian random field
    # ------------------------------------------------------------------

    noise = (
        rng.normal(size=(ngrid, ngrid, ngrid))
        + 1j * rng.normal(size=(ngrid, ngrid, ngrid))
    )

    cell_volume = dx**3

    delta_k = noise * np.sqrt(Pk_grid / (2.0 * cell_volume))

    delta_k[0, 0, 0] = 0.0

    delta = np.real(np.fft.ifftn(delta_k))

    delta -= delta.mean()

    # ------------------------------------------------------------------
    # Galaxy density field
    # ------------------------------------------------------------------

    galaxy_density = nbar * np.maximum(
        0.0,
        1.0 + bias * delta
    )

    expected_counts = galaxy_density * cell_volume

    counts = rng.poisson(expected_counts)

    # ------------------------------------------------------------------
    # Sample galaxy positions
    # ------------------------------------------------------------------

    occupied = np.argwhere(counts > 0)

    positions = []

    for ix, iy, iz in occupied:

        ngal = counts[ix, iy, iz]

        x = (ix + rng.random(ngal)) * dx
        y = (iy + rng.random(ngal)) * dx
        zpos = (iz + rng.random(ngal)) * dx

        positions.append(
            np.column_stack([x, y, zpos])
        )

    if len(positions):
        positions = np.vstack(positions)
    else:
        positions = np.empty((0, 3))

    return positions, delta

In [ ]:
positions, delta = generate_mock_catalogue(
    z=1.0,
    box_size=1000.0,
    ngrid=128,
    nbar=1e-3,
    bias=1.5,
    seed=42
)



In [ ]:
print(positions.shape[0], "galaxies")

plt.scatter(positions[:, 0], positions[:, 1], s=0.1, alpha=0.01)
plt.xlabel("x [Mpc/h]")
plt.ylabel("y [Mpc/h]")
plt.title("Mock Galaxy Catalogue at z=1.0")
plt.show()